el siguiente script de ejemplo entrena un modelo de regresión logística mediante un argumento:
> --regularization  


para establecer el hiperparámetro de tasa de regularización y registra el métrica de precisión con el nombre Accuracy.

In [ ]:
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import mlflow

# get regularization hyperparameter
parser = argparse.ArgumentParser()
parser.add_argument('--regularization', type=float, dest='reg_rate', default=0.01)
args = parser.parse_args([]) # si no pones [] no se leen los valores por defecto
# args es el objeto que contiene los argumentos de la línea de comandos, 
# en este caso solo hay uno: --regularization
reg = args.reg_rate #esta variable extraera el valor de --regularization

# load the training dataset
data = pd.read_csv("./data/iris-data/iris.csv")

# separate features and labels, and split for training/validatiom
X = data.iloc[:, :-1].values #coje las columnas de la 0 a la penultima
#tambien podria ponerse X = data.iloc[:, 0:4].values #coje las columnas de la 0 a la 3
# o data[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
# o data.iloc[:, [0,1,2,3]].values
y = data.iloc[:, -1].values #coje la ultima columna
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30)

# train a logistic regression model with the reg hyperparameter
model = LogisticRegression(C=1/reg, solver="lbfgs", max_iter=1000).fit(X_train, y_train)
#he cambiado el solver de 'liblinear' a 'lbfgs' pq el primero no 
# soporta multinomial y el iris tiene 3 clases (3 valores de target - 3 tipos de flores)

# calculate and log accuracy
y_hat = model.predict(X_test)
acc = np.average(y_hat == y_test)
mlflow.log_metric("Accuracy", acc)

Para preparar el trabajo de barrido, primero debe crear un trabajo base de comando que especifique qué script se va a ejecutar y defina los parámetros usados por el script

In [12]:
from azure.ai.ml import command

# configure command job as base
job = command(
    code="./src",
    command="python train.py --regularization ${{inputs.reg_rate}}",
    inputs={
        "reg_rate": 0.01,
    },
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute="aml-cluster",
)